In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Load dataset
dengue = pd.read_csv("/content/drive/MyDrive/Clean datasets/Dengue_Cleaned.csv")

# Convert date
dengue["ds"] = pd.to_datetime(dengue["ds"])

# Sort by date
dengue = dengue.sort_values("ds").reset_index(drop=True)

# Make sure cases are numeric
dengue["y"] = pd.to_numeric(dengue["y"], errors="coerce")

# Remove missing values
dengue = dengue.dropna(subset=["ds", "y"])

print("Rows:", len(dengue))
print("Start:", dengue["ds"].min())
print("End:", dengue["ds"].max())

dengue.head()

Rows: 37
Start: 2023-06-01 00:00:00
End: 2026-06-01 00:00:00


,ds,y
0,2023-06-01,528
1,2023-07-01,960
2,2023-08-01,602
3,2023-09-01,953
4,2023-10-01,1658


In [3]:
train = dengue.iloc[:-12].copy()
test = dengue.iloc[-12:].copy()

print("Training rows:", len(train))
print("Testing rows:", len(test))

print("\nTraining period:")
print(train["ds"].min(), "to", train["ds"].max())

print("\nTesting period:")
print(test["ds"].min(), "to", test["ds"].max())

Training rows: 25
Testing rows: 12

Training period:
2023-06-01 00:00:00 to 2025-06-01 00:00:00

Testing period:
2025-07-01 00:00:00 to 2026-06-01 00:00:00


In [4]:
dengue_model = ExponentialSmoothing(
    train.set_index("ds")["y"],
    trend=None,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
)

dengue_model_fit = dengue_model.fit(
    optimized=True
)

print("Dengue Holt-Winters model trained successfully.")

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Dengue Holt-Winters model trained successfully.


In [5]:
test_predictions = dengue_model_fit.forecast(
    steps=len(test)
)

test_predictions = np.maximum(
    test_predictions,
    0
)

print(test_predictions)

2025-07-01     614.494413
2025-08-01    2072.939397
2025-09-01    4124.654432
2025-10-01    7240.340451
2025-11-01    3293.745580
2025-12-01    1309.499374
2026-01-01     369.191043
2026-02-01       0.000000
2026-03-01       0.000000
2026-04-01       0.000000
2026-05-01    2346.499182
2026-06-01     476.063265
Freq: MS, dtype: float64


In [6]:
dengue_mae = mean_absolute_error(
    test["y"],
    test_predictions
)

dengue_rmse = np.sqrt(
    mean_squared_error(
        test["y"],
        test_predictions
    )
)

print("Dengue Holt-Winters Validation")
print("--------------------------------")
print("MAE  :", dengue_mae)
print("RMSE :", dengue_rmse)

Dengue Holt-Winters Validation
--------------------------------
MAE  : 771.2298028431142
RMSE : 910.7717160167551


In [7]:
final_dengue_model = ExponentialSmoothing(
    dengue.set_index("ds")["y"],
    trend=None,
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
)

final_dengue_model_fit = final_dengue_model.fit(
    optimized=True
)

print("Final Dengue model trained on all available data.")

Final Dengue model trained on all available data.


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [8]:
future_forecast = final_dengue_model_fit.forecast(
    steps=6
)

future_dates = pd.date_range(
    start=dengue["ds"].max() + pd.offsets.MonthBegin(1),
    periods=6,
    freq="MS"
)

future_dengue = pd.DataFrame({
    "Date": future_dates,
    "Predicted Cases": np.maximum(
        future_forecast.values,
        0
    )
})

future_dengue["Predicted Cases"] = (
    future_dengue["Predicted Cases"]
    .round()
    .astype(int)
)

future_dengue

,Date,Predicted Cases
0,2026-07-01,1318
1,2026-08-01,2166
2,2026-09-01,4312
3,2026-10-01,7865
4,2026-11-01,4388
5,2026-12-01,1735


In [9]:
import joblib

joblib.dump(
    final_dengue_model_fit,
    "dengue_holt_winters_monthly.pkl"
)

print("Dengue forecasting model saved successfully.")

Dengue forecasting model saved successfully.
